# Demo 04 — DSPy GEPA: Reflective Prompt Optimization

**GEPA** (Reflective Prompt Evolution) is a DSPy optimizer that treats the program's instructions as
a variable and searches for a better version using your training examples — not just scalar scores,
but qualitative text feedback about *why* an output failed.

**The optimization loop:**

```
Run program on examples
       ↓
Identify failures → generate text feedback (what went wrong, why)
       ↓
Reflection model analyzes failure patterns across a batch
       ↓
Propose improved instructions targeting those patterns
       ↓
Validate — retain only improvements that increase the metric
       ↓
Repeat (builds a tree of evolved prompt candidates)
```

**Two-model design:**
- *Student model* — fast and cheap; handles all inference (~90–95% of calls)
- *Reflection model* — stronger; analyzes failures and proposes new instructions (~5–10% of calls)

Paper: [GEPA: Reflective Prompt Evolution Can Outperform Reinforcement Learning](https://arxiv.org/abs/2507.19457)

In [1]:
# Dependencies:
# !pip install dspy gepa python-dotenv

## Part 1 — Configure LMs

GEPA requires two separate language models:
- **Student LM** — used for inference on every training example. Keep it fast and cheap.
- **Reflection LM** — used by GEPA to analyze failure patterns and propose new instructions.
  This should be a strong model; it sees failure traces and writes improved prompts.

Both are loaded via OpenRouter here.

In [2]:
import os
import dspy
from dotenv import load_dotenv

load_dotenv()

# Student model: fast and cheap — used for all inference during optimization
student_lm = dspy.LM(
    model="anthropic/claude-haiku-4.5",
    api_key=os.getenv("OPENROUTER_API_KEY"),
    api_base="https://openrouter.ai/api",
    extra_headers={
        "HTTP-Referer": "https://my-app.com",
        "X-Title": "DSPy GEPA Demo",
    },
)

# Reflection model: stronger — used by GEPA to reflect on failures and evolve instructions
reflection_lm = dspy.LM(
    model="anthropic/claude-sonnet-4-6",
    api_key=os.getenv("OPENROUTER_API_KEY"),
    api_base="https://openrouter.ai/api",
    extra_headers={
        "HTTP-Referer": "https://my-app.com",
        "X-Title": "DSPy GEPA Demo",
    },
    temperature=1.0,
    max_tokens=8000,
)

dspy.configure(lm=student_lm)

## Part 2 — Define the Task

Task: classify the sentiment of product reviews as `positive`, `negative`, or `neutral`.

A DSPy `Signature` defines the typed input/output contract. `dspy.Predict` wraps it into a callable
module. The baseline starts with minimal instructions — GEPA will evolve them.

In [3]:
class SentimentClassifier(dspy.Signature):
    """Classify the sentiment of a product review."""
    text: str = dspy.InputField(desc="A product review")
    sentiment: str = dspy.OutputField(desc="Sentiment label: positive, negative, or neutral")

classifier = dspy.Predict(SentimentClassifier)

# Quick sanity check
result = classifier(text="This blender is amazing — smoothies in 30 seconds!")
print(result.sentiment)

positive


## Part 3 — Training and Test Data

12 training examples (4 per class) and 9 test examples (3 per class).
The training set is what GEPA reflects on to evolve the instructions.
The test set is held out to measure improvement.

In [4]:
trainset = [
    # Positive
    dspy.Example(text="This blender is amazing — smoothies in 30 seconds and easy to clean!", sentiment="positive").with_inputs("text"),
    dspy.Example(text="Fast shipping, great quality, exceeded all my expectations.", sentiment="positive").with_inputs("text"),
    dspy.Example(text="Works perfectly out of the box. Highly recommend to everyone.", sentiment="positive").with_inputs("text"),
    dspy.Example(text="Incredible value for money. Far better than more expensive alternatives.", sentiment="positive").with_inputs("text"),
    # Negative
    dspy.Example(text="Broke after two weeks. Total waste of money.", sentiment="negative").with_inputs("text"),
    dspy.Example(text="Nothing like the description. Very disappointed with this purchase.", sentiment="negative").with_inputs("text"),
    dspy.Example(text="Would not recommend. Poor quality and poor customer service.", sentiment="negative").with_inputs("text"),
    dspy.Example(text="Stopped working after 3 uses. Avoid this product.", sentiment="negative").with_inputs("text"),
    # Neutral
    dspy.Example(text="It does what it says. Nothing special, but gets the job done.", sentiment="neutral").with_inputs("text"),
    dspy.Example(text="Average product. Arrived on time, packaging was fine.", sentiment="neutral").with_inputs("text"),
    dspy.Example(text="OK for the price. Some features work well, some don't.", sentiment="neutral").with_inputs("text"),
    dspy.Example(text="Decent product. Serves its purpose, nothing more.", sentiment="neutral").with_inputs("text"),
]

testset = [
    # Positive
    dspy.Example(text="Five stars! This is exactly what I needed.", sentiment="positive").with_inputs("text"),
    dspy.Example(text="Love this product. Will definitely buy again.", sentiment="positive").with_inputs("text"),
    dspy.Example(text="Outstanding quality. I'm very impressed with every aspect.", sentiment="positive").with_inputs("text"),
    # Negative
    dspy.Example(text="Complete rubbish. Threw it out after one day.", sentiment="negative").with_inputs("text"),
    dspy.Example(text="Extremely disappointed. The quality is terrible.", sentiment="negative").with_inputs("text"),
    dspy.Example(text="Doesn't work as described. Very frustrating experience.", sentiment="negative").with_inputs("text"),
    # Neutral
    dspy.Example(text="Product arrived as expected. Functional.", sentiment="neutral").with_inputs("text"),
    dspy.Example(text="Does the job. Nothing extraordinary.", sentiment="neutral").with_inputs("text"),
    dspy.Example(text="It's fine for basic use. Not exceptional, not bad.", sentiment="neutral").with_inputs("text"),
]

print(f"Training examples: {len(trainset)}")
print(f"Test examples:     {len(testset)}")

Training examples: 12
Test examples:     9


## Part 4 — Baseline Evaluation

`dspy.Evaluate` runs the module on each test example, applies the metric, and averages the result.
This is the score GEPA will try to improve.

In [5]:
from dspy.evaluate import Evaluate

def accuracy_metric(gold, pred, trace=None):
    return gold.sentiment.lower().strip() == pred.sentiment.lower().strip()

evaluate = Evaluate(devset=testset, metric=accuracy_metric, num_threads=1, display_progress=True)
baseline_score = evaluate(classifier).score
print(f"\nBaseline accuracy: {baseline_score:.1f}%")

  0%|          | 0/9 [00:00<?, ?it/s]

Average Metric: 1.00 / 1 (100.0%):   0%|          | 0/9 [00:00<?, ?it/s]

Average Metric: 2.00 / 2 (100.0%):  11%|█         | 1/9 [00:00<00:00, 70.09it/s]

Average Metric: 3.00 / 3 (100.0%):  22%|██▏       | 2/9 [00:00<00:00, 102.88it/s]

Average Metric: 4.00 / 4 (100.0%):  33%|███▎      | 3/9 [00:00<00:00, 121.48it/s]

...

Average Metric: 8.00 / 8 (100.0%):  78%|███████▊  | 7/9 [00:00<00:00, 161.39it/s]

Average Metric: 9.00 / 9 (100.0%):  89%|████████▉ | 8/9 [00:00<00:00, 166.41it/s]

Average Metric: 9.00 / 9 (100.0%): 100%|██████████| 9/9 [00:00<00:00, 186.46it/s]

2026/07/06 23:39:00 INFO dspy.evaluate.evaluate: Average Metric: 9 / 9 (100.0%)




Baseline accuracy: 100.0%


## Part 5 — GEPA Metric with Rich Feedback

This is GEPA's key difference from other optimizers. Instead of returning only a score,
the metric can return a `ScoreWithFeedback` — a score paired with a text explanation of the error.

GEPA calls the metric with two extra arguments:
- `pred_name` — the name of the predictor currently being optimized
- `pred_trace` — the execution sub-trace for that predictor

When `pred_name` is set (called by GEPA), return `ScoreWithFeedback` with an explanation.
When `pred_name` is `None` (called by `Evaluate`), return a plain float.

The reflection model reads these feedback strings to identify *patterns* across failures
and propose instructions that address the root cause — not just individual mistakes.

In [6]:
from dspy.teleprompt.gepa.gepa_utils import ScoreWithFeedback

def sentiment_metric(gold, pred, trace=None, pred_name=None, pred_trace=None):
    gold_label = gold.sentiment.lower().strip()
    pred_label = pred.sentiment.lower().strip()
    correct = gold_label == pred_label

    # Called by GEPA: return rich text feedback so the reflection model can analyze patterns
    if pred_name is not None:
        if correct:
            return ScoreWithFeedback(score=1.0, feedback="Correct classification.")
        return ScoreWithFeedback(
            score=0.0,
            feedback=(
                f"Wrong. The review '{gold.text}' expresses '{gold_label}' sentiment, "
                f"not '{pred_label}'. "
                f"Consider: overall emotional tone, specific word choices "
                f"(e.g. 'amazing', 'disappointed', 'decent'), and whether the reviewer "
                f"conveys enthusiasm, frustration, or neutral observation."
            ),
        )

    # Called by Evaluate: return a plain float
    return 1.0 if correct else 0.0

## Part 6 — Run GEPA Optimization

Key parameters:
- `reflection_lm` — the stronger model that reflects on failures and proposes new instructions
- `max_metric_calls` — budget cap on total metric calls. Set small here for a fast demo.
  In production, use `auto="medium"` or `auto="heavy"` for a larger budget.
- `reflection_minibatch_size` — number of examples analyzed per reflection step
- `trainset` — examples GEPA reflects on to propose improvements
- `valset` — examples used to track the Pareto frontier of candidate instructions

> `dspy.GEPA` is marked `@experimental` in DSPy 3.x and requires the `gepa` package.
> This run uses `max_metric_calls=60` to keep the demo quick (~2–3 minutes).

In [7]:
gepa = dspy.GEPA(
    metric=sentiment_metric,
    reflection_lm=reflection_lm,
    max_metric_calls=60,
    reflection_minibatch_size=3,
    num_threads=1,
)

optimized_classifier = gepa.compile(
    student=dspy.Predict(SentimentClassifier),
    trainset=trainset,
    valset=testset,
)

2026/07/06 23:39:00 INFO dspy.teleprompt.gepa.gepa: Running GEPA for approx 60 metric calls of the program. This amounts to 2.86 full evals on the train+val set.


2026/07/06 23:39:00 INFO dspy.teleprompt.gepa.gepa: Using 9 examples for tracking Pareto scores.


GEPA Optimization:   0%|          | 0/60 [00:00<?, ?rollouts/s]

2026/07/06 23:39:00 INFO dspy.evaluate.evaluate: Average Metric: 9.0 / 9 (100.0%)


2026/07/06 23:39:00 INFO dspy.teleprompt.gepa.gepa: Iteration 0: Base program full valset score: 1.0 over 9 / 9 examples


...

Average Metric: 3.00 / 3 (100.0%): 100%|██████████| 3/3 [00:00<00:00, 169.01it/s]

2026/07/06 23:39:21 INFO dspy.evaluate.evaluate: Average Metric: 3.0 / 3 (100.0%)


2026/07/06 23:39:21 INFO dspy.teleprompt.gepa.gepa: Iteration 17: All subsample scores perfect. Skipping.


2026/07/06 23:39:21 INFO dspy.teleprompt.gepa.gepa: Iteration 17: Reflective mutation did not propose a new candidate


GEPA Optimization:  95%|█████████▌| 57/60 [00:21<00:01,  2.69rollouts/s]

## Part 7 — Compare Before and After

GEPA evolves the *instructions* field of each predictor's signature.
Inspect what changed, then measure accuracy on the held-out test set.

In [8]:
# Show how the instructions evolved
print("=== Original instructions ===")
print(classifier.signature.instructions)

print("\n=== Optimized instructions (GEPA) ===")
for name, predictor in optimized_classifier.named_predictors():
    print(f"[{name}]")
    print(predictor.signature.instructions)

=== Original instructions ===
Classify the sentiment of a product review.

=== Optimized instructions (GEPA) ===
[self]
Classify the sentiment of a product review.


In [9]:
# Measure accuracy on the test set
optimized_score = evaluate(optimized_classifier).score

print(f"Baseline accuracy:       {baseline_score:.1f}%")
print(f"GEPA optimized accuracy: {optimized_score:.1f}%")
print(f"Improvement:             {optimized_score - baseline_score:+.1f}%")

  0%|          | 0/9 [00:00<?, ?it/s]

Average Metric: 1.00 / 1 (100.0%):   0%|          | 0/9 [00:00<?, ?it/s]

Average Metric: 2.00 / 2 (100.0%):  11%|█         | 1/9 [00:00<00:00, 96.41it/s]

Average Metric: 3.00 / 3 (100.0%):  22%|██▏       | 2/9 [00:00<00:00, 125.44it/s]

Average Metric: 4.00 / 4 (100.0%):  33%|███▎      | 3/9 [00:00<00:00, 139.00it/s]

...

Average Metric: 8.00 / 8 (100.0%):  78%|███████▊  | 7/9 [00:00<00:00, 160.57it/s]

Average Metric: 9.00 / 9 (100.0%):  89%|████████▉ | 8/9 [00:00<00:00, 164.92it/s]

Average Metric: 9.00 / 9 (100.0%): 100%|██████████| 9/9 [00:00<00:00, 184.89it/s]

2026/07/06 23:39:22 INFO dspy.evaluate.evaluate: Average Metric: 9 / 9 (100.0%)



Baseline accuracy:       100.0%
GEPA optimized accuracy: 100.0%
Improvement:             +0.0%


## Further Reading

- [DSPy GEPA tutorial — AI program optimization](https://dspy.ai/tutorials/gepa_ai_program/)
- [DSPy GEPA tutorial — AIME benchmark](https://dspy.ai/tutorials/gepa_aime/)
- [HuggingFace cookbook — DSPy GEPA](https://huggingface.co/learn/cookbook/en/dspy_gepa)
- [GEPA paper: Reflective Prompt Evolution Can Outperform Reinforcement Learning](https://arxiv.org/abs/2507.19457)
- [GEPA GitHub](https://github.com/gepa-ai/gepa)